# LangGraph Customer Support Agent with Snowflake AI Observability

This notebook demonstrates how to:

1. **Build a LangGraph agent** that uses Snowflake Cortex Search, Cortex Analyst, and chart generation as tools
2. **Instrument the agent with TruLens** (`TruGraph`) for full trace capture
3. **Run evaluations** using Snowflake AI Observability to measure answer relevance, context relevance, and groundedness

### Architecture

```
User Query
    │
    ▼
LangGraph ReAct Agent (ChatSnowflake LLM)
    ├── Tool: Cortex Search (unstructured case lookups)
    ├── Tool: Cortex Analyst (structured metrics via semantic view)
    └── Tool: Generate Chart (Vega-Lite visualizations via Cortex LLM)
    │
    ▼
TruGraph (auto-instrumentation) → Snowflake AI Observability
```

### Prerequisites

- Run `setup.sql` to create the `CUST_SUPPORT_DEMO.SUPPORT` schema, tables, semantic view, and Cortex Search service
- Snowflake account with Cortex features enabled
- `CORTEX_USER` database role and `CREATE EXTERNAL AGENT` / `CREATE TASK` / `EXECUTE TASK` privileges

In [1]:
%pip install --quiet -U langchain-core langchain-snowflake langgraph trulens-core trulens-providers-cortex trulens-connectors-snowflake trulens-apps-langgraph altair

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.38.0 requires altair<6,>=4.0, but you have altair 6.0.0 which is incompatible.
streamlit-aggrid 1.0.5 requires altair<5, but you have altair 6.0.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## 1. Snowflake Session Setup

In [2]:
import os

sorted(os.environ)[-30:]

['PYTHONUNBUFFERED',
 'PYTHON_FROZEN_MODULES',
 'SHELL',
 'SHLVL',
 'SNOWFLAKE_ACCOUNT',
 'SNOWFLAKE_FORCE_GCP_USE_DOWNSCOPED_CREDENTIAL',
 'SNOWFLAKE_PASSWORD',
 'SNOWFLAKE_PAT',
 'SNOWFLAKE_USER',
 'SNOWFLAKE_USER_PASSWORD',
 'SSH_AUTH_SOCK',
 'TERM',
 'TMPDIR',
 'USER',
 'VSCODE_CODE_CACHE_PATH',
 'VSCODE_CRASH_REPORTER_PROCESS_TYPE',
 'VSCODE_CWD',
 'VSCODE_ESM_ENTRYPOINT',
 'VSCODE_HANDLES_UNCAUGHT_ERRORS',
 'VSCODE_IPC_HOOK',
 'VSCODE_L10N_BUNDLE_LOCATION',
 'VSCODE_NLS_CONFIG',
 'VSCODE_PID',
 'XPC_FLAGS',
 'XPC_SERVICE_NAME',
 '_',
 '_CE_CONDA',
 '_CE_M',
 '__CFBundleIdentifier',
 '__CF_USER_TEXT_ENCODING']

In [3]:
import os

#Define env variables as necessary 

# os.environ["SNOWFLAKE_ACCOUNT"] = "<your_account>"
# os.environ["SNOWFLAKE_USER"] = "<your_username>"
# os.environ["SNOWFLAKE_PASSWORD"] = "<your_password>"
os.environ["SNOWFLAKE_WAREHOUSE"] = "COMPUTE_WH"
os.environ["SNOWFLAKE_DATABASE"] = "CUST_SUPPORT_DEMO"
os.environ["SNOWFLAKE_SCHEMA"] = "AGENTS"

from langchain_snowflake import create_session_from_env

session = create_session_from_env()
print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

/opt/anaconda3/envs/snow_tru_py314/lib/python3.11/site-packages/langchain_snowflake/chat_models/base.py:26: UserWarning: Field name "schema" in "ChatSnowflake" shadows an attribute in parent "BaseChatModel"
  class ChatSnowflake(


Connected: "CUST_SUPPORT_DEMO"."AGENTS"


## 2. Define Snowflake Cortex Tools

In [4]:
import json
import textwrap
from langchain_core.tools import tool
from langchain_snowflake import SnowflakeCortexSearchRetriever

from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes

CORTEX_SEARCH_SERVICE = "CUST_SUPPORT_DEMO.AGENTS.CASE_SEARCH_SERVICE"

retriever = SnowflakeCortexSearchRetriever(
    session=session,
    schema="CUST_SUPPORT_DEMO.AGENTS",
    service_name=CORTEX_SEARCH_SERVICE,
    k=10,
    auto_format_for_rag=True,
    content_field="ISSUE_SUMMARY",
    join_separator="\n\n",
    fallback_to_page_content=True,
)


@tool
@instrument(
    span_type=SpanAttributes.SpanType.RETRIEVAL,
    attributes={
        SpanAttributes.RETRIEVAL.QUERY_TEXT: "query",
        SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
    },
)
def search_support_cases(query: str) -> list:
    """ Search specific case details, issue descriptions, and resolution summaries by topic or keyword.
        Returns CASE_ID, CUSTOMER_ID, PRODUCT, ISSUE_CATEGORY, PRIORITY, STATUS, REP_NAME as attributes.
        Always cite CASE_ID values in responses. NOT for aggregate metrics or trends"""
    docs = retriever.invoke(query)
    # return "\n\n".join(doc.page_content for doc in docs)
    return [doc.page_content for doc in docs]

print("Cortex Search tool defined.")
test_results = search_support_cases.invoke("authentication login failures")
# print(textwrap.shorten(test_results, width=200, placeholder="..."))

Cortex Search tool defined.


In [5]:
import requests

SEMANTIC_VIEW = "CUST_SUPPORT_DEMO.AGENTS.SUPPORT_ANALYTICS"


@tool
@instrument(
    span_type=SpanAttributes.SpanType.RETRIEVAL,
    attributes={
        SpanAttributes.RETRIEVAL.QUERY_TEXT: "question",
        SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
    },
)
def query_support_analytics(question: str) -> list:
    """ Quantitative support metrics: case counts, CSAT scores, resolution times, first response
        times, escalation rates, rep performance, daily/weekly trends, breakdowns by product/category/priority/rep.
        Use for aggregation, filtering, or numerical analysis. NOT for case descriptions or resolutions."""
    host = session.connection.host
    token = session.connection.rest.token
    # token = os.getenv("SNOWFLAKE_PAT")

    resp = requests.post(
        url=f"https://{host}/api/v2/cortex/analyst/message",
        json={
            "messages": [
                {"role": "user", "content": [{"type": "text", "text": question}]}
            ],
            "semantic_view": SEMANTIC_VIEW,
        },
        headers={
            "Authorization": f'Snowflake Token="{token}"',
            "Content-Type": "application/json",
        },
    )
    resp.raise_for_status()
    data = resp.json()

    result_parts = []
    msg = data.get("message") or {}
    for block in msg.get("content", []):
        if block.get("type") == "text":
            result_parts.append(block["text"])
        elif block.get("type") == "sql":
            result_parts.append(f"SQL: {block['statement']}")
            sql_results = session.sql(block["statement"]).collect()
            result_parts.append(json.dumps([row.as_dict() for row in sql_results[:20]], default=str))
        elif block.get("type") == "result_table":
            result_parts.append(json.dumps(block.get("data", [])[:20], default=str))
    # return "\n".join(result_parts) if result_parts else "No results returned."
    return result_parts

print("Cortex Analyst tool defined.")
test_analytics = query_support_analytics.invoke("How many total support cases are there?")
print(test_analytics[:300])

Cortex Analyst tool defined.
['This is our interpretation of your question:\n\nHow many total support cases are there over the entire available time period?', 'SQL: WITH __case_metrics AS (\n  SELECT\n    case_id,\n    case_date\n  FROM CUST_SUPPORT_DEMO.AGENTS.CASE_METRICS\n)\nSELECT\n  MIN(case_date) AS start_date,\n  MAX(case_date) AS end_date,\n  COUNT(case_id) AS total_cases\nFROM __case_metrics\n -- Generated by Cortex Analyst (request_id: dc96d800-54e9-4b98-8801-464bff18cc4b)\n;', '[{"START_DATE": "2025-12-01", "END_DATE": "2026-02-28", "TOTAL_CASES": 200}]']


In [38]:
import altair as alt
from snowflake.cortex import Complete


@tool
def generate_chart(data: list[dict], chart_description: str) -> str:
    """Generate a Vega-Lite chart visualization from data.
    Use after query_support_analytics returns tabular results that would benefit
    from visual representation (trends, comparisons, distributions, rankings).

    Args:
        data: List of dictionaries representing rows of data (from SQL query results).
        chart_description: Natural language description of the desired chart.
    """
    prompt = f"""Generate a Vega-Lite v5 JSON specification for the following:
Description: {chart_description}
Data: {json.dumps(data[:50], default=str)}

Return ONLY valid JSON with $schema set to "https://vega.github.io/schema/vega-lite/v5.json".
Embed the data inline using the "data.values" field. Include title, appropriate mark type,
and encoding. Use a clean, professional style with a width of 500 and height of 300."""

    spec_text = Complete(
        model="claude-4-sonnet",
        prompt=[{"role": "user", "content": prompt}],
        session=session,
    )

    if "```" in spec_text:
        spec_text = spec_text.split("```")[1]
        if spec_text.startswith("json"):
            spec_text = spec_text[4:]
        spec_text = spec_text.strip()

    chart_spec = json.loads(spec_text)

    chart = alt.Chart.from_dict(chart_spec)
    display(chart)

    return json.dumps(chart_spec)


print("Chart generation tool defined.")

Chart generation tool defined.


## 3. Build the LangGraph Agent

In [39]:
from langchain_snowflake import ChatSnowflake
from langgraph.prebuilt import create_react_agent

llm = ChatSnowflake(
    session=session,
    # model="claude-4-sonnet",
    model="claude-sonnet-4-6",
    temperature=0,
    max_tokens=2048,
)

SYSTEM_PROMPT = """
instructions:
  response: >
    Concise customer support analytics assistant. Be data-driven, use appropriate number formatting.
    ALWAYS cite case IDs (e.g., CASE-00016) from search results. For rankings/comparisons, show the
    COMPLETE list. Group duplicate search results by issue type with counts and representative case IDs.
    Note data limitations when results appear incomplete.
  orchestration: >
    Tool Selection: query_support_analytics for metrics, trends, counts, averages, comparisons.
    search_support_cases for specific case details, issue descriptions, resolutions, or topic searches.
    generate_chart for visualizing tabular data from query_support_analytics (trends, comparisons,
    distributions, rankings). Use BOTH search and analytics tools when a question needs metrics AND
    case examples.

    search_support_cases: ALWAYS include CASE_ID from results. Group duplicate results by issue type
    with representative case IDs rather than repeating identical entries.

    query_support_analytics: For trends, request the FULL date range (data covers Dec 2025-Feb 2026).
    For rankings, return ALL items with values, not just the top result.

    generate_chart: After getting tabular data from query_support_analytics, call generate_chart with
    the raw data rows and a description of the desired visualization. Use for any result with 3+ rows
    that shows trends, comparisons, or distributions.
  sample_questions:
    - question: "What is the average CSAT score by product?"
      answer: "I'll query the support analytics data to get average CSAT scores for all products, present the complete ranking, and generate a bar chart."
    - question: "Find cases related to authentication issues"
      answer: "I'll search the case database for authentication-related cases and cite specific case IDs grouped by issue type."
    - question: "Which rep has the best resolution time?"
      answer: "I'll query rep performance data to rank all reps by average resolution time from fastest to slowest."
    - question: "Show me the trend of daily escalations"
      answer: "I'll query the daily support metrics for the full available date range and generate a line chart showing the escalation trend over time."""

tools = [search_support_cases, query_support_analytics, generate_chart]

graph = create_react_agent(
    llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)

print(f"LangGraph agent compiled. Nodes: {list(graph.get_graph().nodes.keys())}")

LangGraph agent compiled. Nodes: ['__start__', 'agent', 'tools', '__end__']


## 4. Test the Agent

In [40]:
response = graph.invoke(
    {"messages": [("user", "What is the average CSAT score by product?")]}
)
print(response["messages"][-1].content)

/var/folders/kt/jnpc5l790719m738576fttpm0000gn/T/ipykernel_7357/3730780384.py:23: DeprecationWarning: Complete() is deprecated and will be removed in a future release. Use complete() instead
  spec_text = Complete(
/var/folders/kt/jnpc5l790719m738576fttpm0000gn/T/ipykernel_7357/3730780384.py:23: DeprecationWarning: Complete() is deprecated and will be removed in a future release. Use complete() instead
  spec_text = Complete(


alt.Chart(...)

Here are the **average CSAT scores by product** (scale of 1–5), ranked highest to lowest:

| Rank | Product | Avg CSAT |
|------|---------|----------|
| 1 | **AnalyticsHub** | 3.89 |
| 2 | SecureVault | 3.84 |
| 3 | API Gateway | 3.82 |
| 4 | DataSync Pro | 3.79 |
| 5 | CloudStore Platform | 3.77 |

**Key Takeaways:**
- 📊 The scores are **tightly clustered** within a narrow range of **3.77–3.89**, suggesting relatively consistent customer satisfaction across products.
- 🏆 **AnalyticsHub** leads with the highest CSAT of **3.89**.
- ⚠️ **CloudStore Platform** sits at the bottom with **3.77**, though the gap from the top is only **0.12 points**.

Overall, all products are performing at a **moderate satisfaction level** — there may be room for improvement across the board to push scores closer to 5.


In [41]:
response = graph.invoke(
    {"messages": [("user", "plot a bar chart of the average CSAT score by product")]}
)
print(response["messages"][-1].content)

/var/folders/kt/jnpc5l790719m738576fttpm0000gn/T/ipykernel_7357/3730780384.py:23: DeprecationWarning: Complete() is deprecated and will be removed in a future release. Use complete() instead
  spec_text = Complete(
/var/folders/kt/jnpc5l790719m738576fttpm0000gn/T/ipykernel_7357/3730780384.py:23: DeprecationWarning: Complete() is deprecated and will be removed in a future release. Use complete() instead
  spec_text = Complete(


alt.Chart(...)

Here's the bar chart of average CSAT scores by product! Here's a summary of the rankings:

| Rank | Product | Avg CSAT |
|------|---------|----------|
| 1 | **AnalyticsHub** | 3.89 |
| 2 | SecureVault | 3.84 |
| 3 | API Gateway | 3.82 |
| 4 | DataSync Pro | 3.79 |
| 5 | CloudStore Platform | 3.77 |

**Key Takeaways:**
- All products score within a **narrow band** (3.77–3.89) out of 5, suggesting consistently moderate satisfaction across the board.
- **AnalyticsHub** leads with the highest CSAT at **3.89**, while **CloudStore Platform** trails slightly at **3.77**.
- The gap between the top and bottom is only **0.12 points**, indicating no single product is dramatically outperforming or underperforming the others.


In [8]:
response = graph.invoke(
    {"messages": [("user", "Find cases related to authentication failures and how they were resolved")]}
)
print(response["messages"][-1].content)

Here's a summary of authentication failure cases, grouped by issue type:

---

### 🔐 Authentication Failure Cases

#### 1. MFA Token Not Accepted After Password Reset
**9 cases** — This was the most common authentication issue.

- **Issue:** Customers unable to log in after a password reset; MFA tokens not accepted despite correct entry.
- **Resolution:**
  1. Reset MFA configuration and re-enrolled the user.
  2. Cleared cached authentication tokens on the server side.
  3. Verified successful login after re-enrollment.

> ⚠️ *Note: The search returned 9 identical entries for this issue — case IDs were not included in the results. The actual number of distinct cases may vary.*

---

#### 2. SSO / SAML Assertion Errors (Enterprise)
**1 case**

- **Issue:** SSO integration failing with SAML assertion errors for an enterprise account.
- **Resolution:**
  1. Updated an **expired SAML certificate**.
  2. Reconfigured the IdP metadata URL.
  3. Validated the assertion consumer service (ACS)

## 5. Instrument with TruLens + Snowflake AI Observability

We wrap the LangGraph agent with `TruGraph` which:
- Auto-instruments all graph nodes and tool calls
- Captures full execution traces (inputs, outputs, latency)
- Exports traces to Snowflake for the AI Observability UI

In [13]:
#drop external app as necessary
# APP_NAME = "CUSTOMER_SUPPORT_AGENT_LANGGRAPH"

# session.sql(f'DROP EXTERNAL AGENT {APP_NAME}').collect()

In [14]:
from trulens.apps.langgraph import TruGraph
from trulens.connectors.snowflake import SnowflakeConnector

tru_snowflake_connector = SnowflakeConnector(snowpark_session=session)

APP_NAME = "CUSTOMER_SUPPORT_AGENT_LANGGRAPH"
APP_VERSION = "LANGGRAPH_V1"

tru_graph = TruGraph(
    graph,
    app_name=APP_NAME,
    app_version=APP_VERSION,
    connector=tru_snowflake_connector,
    main_method = graph.invoke
)

print(f"TruGraph registered: {APP_NAME} / {APP_VERSION}")

Running the TruLens dashboard requires providing a `password` to the `SnowflakeConnector`.


✅ experimental Feature.OTEL_TRACING enabled.
🔒 experimental Feature.OTEL_TRACING is enabled and cannot be changed.


/opt/anaconda3/envs/snow_tru_py314/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


instrumenting <class 'langgraph.graph.state.StateGraph'> for base <class 'langgraph.graph.state.StateGraph'>
instrumenting <class 'langgraph.graph.state.CompiledStateGraph'> for base <class 'langgraph.graph.state.CompiledStateGraph'>
	instrumenting invoke
	instrumenting ainvoke
	instrumenting stream
	instrumenting astream
instrumenting <class 'langgraph.graph.state.CompiledStateGraph'> for base <class 'langgraph.pregel.main.Pregel'>
	instrumenting invoke
	instrumenting ainvoke
	instrumenting stream
	instrumenting astream
TruGraph registered: CUSTOMER_SUPPORT_AGENT_LANGGRAPH / LANGGRAPH_V1


## 6. Define Evaluation Dataset

We create a test dataset that exercises both tools across different query types.

In [15]:
import pandas as pd

queries_df = session.table("CUST_SUPPORT_DEMO.AGENTS.EVAL_DATA").to_pandas()

queries_df['query_json'] = queries_df['INPUT_QUERY'].apply(lambda x: {"messages": [("user", x)]})
queries_df['ground_truth_string'] = queries_df['GROUND_TRUTH_DATA'].apply(lambda x: json.loads(x).get('ground_truth_output'))
print(f"Evaluation dataset: {len(queries_df)} queries")
queries_df

Evaluation dataset: 10 queries


,INPUT_QUERY,GROUND_TRUTH_DATA,query_json,ground_truth_string
0,What is the average CSAT score by product?,"{\n ""ground_truth_output"": ""The average CSAT ...","{'messages': [('user', 'What is the average CS...",The average CSAT scores by product are: Analyt...
1,Which rep has the fastest average resolution t...,"{\n ""ground_truth_output"": ""Rachel Foster has...","{'messages': [('user', 'Which rep has the fast...",Rachel Foster has the fastest average resoluti...
2,What is the escalation rate for Critical prior...,"{\n ""ground_truth_output"": ""The escalation ra...","{'messages': [('user', 'What is the escalation...",The escalation rate for Critical priority case...
3,Show me the daily trend of total cases over th...,"{\n ""ground_truth_output"": ""The daily total c...","{'messages': [('user', 'Show me the daily tren...",The daily total cases from late January to lat...
4,How many cases were there for each issue categ...,"{\n ""ground_truth_output"": ""Case counts by is...","{'messages': [('user', 'How many cases were th...",Case counts by issue category: Login/Authentic...
5,Find cases related to authentication or login ...,"{\n ""ground_truth_output"": ""Relevant cases in...","{'messages': [('user', 'Find cases related to ...",Relevant cases include: CASE-00016 and CASE-00...
6,What resolutions were used for data loss issues?,"{\n ""ground_truth_output"": ""Data loss cases i...","{'messages': [('user', 'What resolutions were ...",Data loss cases include: CASE-00007 about data...
7,Search for cases involving API integration pro...,"{\n ""ground_truth_output"": ""API integration c...","{'messages': [('user', 'Search for cases invol...",API integration cases include: CASE-00009 abou...
8,What billing issues have customers reported an...,"{\n ""ground_truth_output"": ""Billing issues in...","{'messages': [('user', 'What billing issues ha...",Billing issues include: CASE-00015 about a cus...
9,What is the average first response time by pri...,"{\n ""ground_truth_output"": ""Average first res...","{'messages': [('user', 'What is the average fi...",Average first response time by priority: Low: ...


## 7. Execute Evaluation Run

In [ ]:
from trulens.core.run import Run
from trulens.core.run import RunConfig
import datetime

run_config = RunConfig(
    run_name="LANGGRAPH_SUPPORT_AGENT_EVAL_RUN",
    description="Questions for Support Agent",
    dataset_name="TEST_QUERIES_DF",
    source_type="DATAFRAME",
    label="LANGGRAPH",
    dataset_spec={
        "RECORD_ROOT.INPUT": "query_json",
        "RECORD_ROOT.GROUND_TRUTH_OUTPUT": "ground_truth_string",

    },
    
)

In [17]:
run = tru_graph.add_run(run_config=run_config)

run.start(input_df=queries_df)

Evaluator thread encountered an error: 'str' object has no attribute 'get'
Error in validate connection parameters: Account must be a non-empty string
Error in create Snowflake session: Account must be a non-empty string
Cortex Search REST API failed: Account must be a non-empty string


## 8. Compute Evaluation Metrics

We compute three key RAG evaluation metrics:
- **Answer Relevance**: Is the response relevant to the user's query?
- **Context Relevance**: Is the retrieved context relevant to the query?
- **Groundedness**: Is the response grounded in the retrieved context?

In [19]:
import time

metric_list = [
    "correctness",
    "groundedness",
    "coherence",
]

run.compute_metrics(metric_list)
print("Metrics computation started...")

while run.get_status() not in ("COMPLETED", "PARTIALLY_COMPLETED", "CANCELLED"):
    print(f"  Status: {run.get_status()}")
    time.sleep(5)

print(f"Evaluation complete. Final status: {run.get_status()}")

Metrics computation started...
  Status: RunStatus.INVOCATION_COMPLETED
  Status: RunStatus.INVOCATION_COMPLETED
  Status: RunStatus.INVOCATION_COMPLETED
  Status: RunStatus.INVOCATION_COMPLETED
  Status: RunStatus.COMPUTATION_IN_PROGRESS
  Status: RunStatus.COMPUTATION_IN_PROGRESS
  Status: RunStatus.COMPUTATION_IN_PROGRESS
  Status: RunStatus.COMPUTATION_IN_PROGRESS
  Status: RunStatus.COMPUTATION_IN_PROGRESS
  Status: RunStatus.COMPUTATION_IN_PROGRESS
  Status: RunStatus.COMPUTATION_IN_PROGRESS
Evaluation complete. Final status: RunStatus.COMPLETED


In [15]:
f"batch_run_{run_version}"

## 9. View Results

Results are available in the **Snowsight AI Observability UI**:

1. Navigate to **AI & ML > Evaluations** in Snowsight
2. Filter by app name: `customer_support_langgraph_agent`
3. Select the run to inspect individual traces and metrics

You can also query the results programmatically:

In [24]:
print(f"Run Name:   {run_config.run_name}")
print(f"App Name:   {APP_NAME}")
print(f"App Version: {APP_VERSION}")
print(f"Final Status: {run.get_status()}")
print()
print("View full results in Snowsight:")
print("  → AI & ML > Evaluations")
print(f"  → Look for app '{APP_NAME}' version '{APP_VERSION}'")

In [29]:
eval_df = session.sql(f"""
    SELECT *
    FROM TABLE(
        SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
            'CUST_SUPPORT_DEMO', 'SUPPORT', '{APP_NAME}', 'EXTERNAL AGENT', '{run_config.run_name}'
        )
    )
    ORDER BY TIMESTAMP DESC
    LIMIT 50
""").to_pandas()

print(f"Evaluation records: {len(eval_df)}")
if not eval_df.empty:
    display(eval_df)
else:
    print("No evaluation data yet. Results may take a moment to appear.")
    print("Check Snowsight → AI & ML → Evaluations for the latest results.")

In [32]:
obs_event_sql = f"""SELECT * FROM TABLE(
    SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
        'CUST_SUPPORT_DEMO', 
        'SUPPORT', 
        '{APP_NAME}', 
        'EXTERNAL AGENT')
) 
WHERE RECORD_ATTRIBUTES:"snow.ai.observability.run.name" = '{run_config.run_name}';"""

df = session.sql(obs_event_sql).to_pandas()

df

In [ ]:
df

## Next Steps

- **Iterate on the agent**: Try different LLMs (`llama3.1-70b`, `mistral-large`), adjust the system prompt, or add more tools
- **Compare versions**: Create new app versions and compare evaluation runs side-by-side in Snowsight
- **Add ground truth**: Include expected answers in the dataset to compute the `correctness` metric
- **Add more metrics**: Compute `coherence` or custom metrics for domain-specific evaluation
- **Production deployment**: Use the best-performing configuration to deploy as a Cortex Agent or Streamlit app